# 08 · New Columns

Making new columns from old ones: plain column math, assign with pd.col, and why apply is slow.

In [1]:
# Setup: the steps from the previous parts (run this first)
import pandas as pd
pd.set_option("display.max_colwidth", 30)

URL = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
       "master/data/2021/2021-04-20/netflix_titles.csv")
df = pd.read_csv(URL)

df = df.drop(columns=["cast", "description"])

df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df = df.dropna(subset=["date_added", "rating"])

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()
df["duration_num"] = df["duration"].str.split(" ").str[0].astype(int)
df["main_country"] = df["country"].str.split(", ").str[0]

In [2]:
df["years_to_netflix"] = df["year_added"] - df["release_year"]
df[["title", "release_year", "year_added", "years_to_netflix"]].head()

,title,release_year,year_added,years_to_netflix
0,3%,2020,2020,0
1,7:19,2016,2016,0
2,23:59,2011,2018,7
3,9,2009,2017,8
4,21,2008,2020,12


In [3]:
movies = df[df["type"] == "Movie"].assign(hours=pd.col("duration_num") / 60)
movies[["title", "duration_num", "hours"]].head(3)

,title,duration_num,hours
1,7:19,93,1.550000
2,23:59,78,1.300000
3,9,80,1.333333


In [4]:
%timeit df["release_year"].apply(lambda year: year + 1)

6.19 ms ± 2.14 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%timeit df["release_year"] + 1

171 μs ± 55.4 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
